[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/02_Linear_vs_Complex_Relationships.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/02_Linear_vs_Complex_Relationships.ipynb)

# Is the Relationship Linear? — Visual Tests Before You Model

A linear model draws one straight line through the data.
If the truth is a curve, a jump, or a combination of two features — a straight line is wrong by design.

The most important question before choosing a model:
**Does the target change proportionally with each feature?**

---

**What this notebook covers**

1. What a linear relationship looks like in a scatter plot
2. Three ways the truth can be non-linear: curved, threshold, interaction
3. The residual test — the most reliable way to check if linear is enough
4. How a decision tree handles what a linear model cannot

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

np.random.seed(0)
n = 250
x  = np.linspace(1, 10, n) + np.random.normal(0, 0.1, n)
x2 = np.random.uniform(1, 5, n)
noise = np.random.normal(0, 0.6, n)
print('Ready.')

## Part 1 — Four Datasets, Four Relationship Shapes

We build four synthetic datasets, each with a **known** true rule.

| Dataset | True rule | What the scatter looks like |
|---|---|---|
| A — Linear | y = 2x + noise | Straight tilted band |
| B — Curved | y = (x−5.5)² + noise | U-shape / parabola |
| C — Threshold | y = 0 if x < 5 else 10 + noise | Two horizontal bands with a jump |
| D — Interaction | y = x₁ × x₂ + noise | No clear pattern with x₁ alone |

A linear model can only describe Dataset A correctly.

In [ ]:
yA = 2*x + noise
yB = (x - 5.5)**2 + noise
yC = np.where(x < 5, 0, 10) + noise
yD = x * x2 + noise

datasets = [
    (x, yA, 'A — Linear\ny = 2x + noise',           '#2a6f97'),
    (x, yB, 'B — Curved\ny = (x-5.5)² + noise',     '#26A69A'),
    (x, yC, 'C — Threshold\ny = 0 if x<5 else 10',  '#FFA726'),
    (x, yD, 'D — Interaction\ny = x1 × x2 + noise', '#AB47BC'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (xi, yi, title, col) in zip(axes, datasets):
    ax.scatter(xi, yi, s=12, alpha=0.45, color=col)
    ax.set_xlabel('x (feature)')
    ax.set_title(title, fontsize=10)
axes[0].set_ylabel('y (target)')
plt.suptitle('Look at the scatter plots first — what shape is the relationship?', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('A: tilted straight band — linear is correct')
print('B: U-shape — linear cannot follow a curve')
print('C: two flat bands with a jump — linear cannot make a sharp step')
print('D: no clear pattern with x1 alone — the pattern requires both x1 and x2 together')

## Part 2 — The Residual Test

Fitting a linear model and examining its **residuals** is the most reliable test.

```
residual = actual value − model's prediction
```

**Random residuals scattered around zero** → linear captured the pattern correctly.
**Residuals that form a shape** (curve, wave, two clusters) → the linear model missed something systematic.

Shaped residuals mean the model's errors are not random — they are predictable.
And if errors are predictable, a better model could reduce them.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col_idx, (xi, yi, title, color) in enumerate(datasets):
    X_col = xi.reshape(-1, 1)
    lm    = LinearRegression().fit(X_col, yi)
    pred  = lm.predict(X_col)
    resid = yi - pred
    r     = rmse(yi, pred)

    sort_i = xi.argsort()

    ax_top = axes[0, col_idx]
    ax_top.scatter(xi, yi, s=10, alpha=0.35, color=color)
    ax_top.plot(xi[sort_i], pred[sort_i], 'k-', lw=2, label='Linear fit')
    ax_top.set_title(f'{title}\nRMSE = {r:.2f}', fontsize=9)
    ax_top.set_xlabel('x')
    if col_idx == 0:
        ax_top.set_ylabel('y (target)')

    ax_bot = axes[1, col_idx]
    ax_bot.scatter(xi, resid, s=10, alpha=0.35, color=color)
    ax_bot.axhline(0, color='black', lw=1.5, ls='--')
    ax_bot.set_xlabel('x')
    ax_bot.set_title('Residuals\n(actual − predicted)', fontsize=9)
    if col_idx == 0:
        ax_bot.set_ylabel('Residual')

plt.suptitle('Linear fit (top) and residuals (bottom)\nRandom residuals = OK. Shaped residuals = need a different model', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print('A: residuals look like random noise around zero — linear is working')
print('B: residuals form a U-shape — the model always over-predicts at the ends')
print('C: residuals form two tilted bands — the model missed the jump')
print('D: residuals have structure — x2 (hidden feature) is creating a pattern the model never saw')

## Part 3 — A Decision Tree Handles What Linear Cannot

A decision tree learns rules of the form:
- "if x ≤ 4.9 → predict 0.1"
- "if x > 4.9 → predict 9.8"

These rules capture **thresholds** directly.
By using many splits, a tree can also **approximate curves** with small steps.
And with two features, it can capture **interactions** by splitting on each feature separately.

A linear model has one formula for all observations. A tree has different formulas for different regions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

non_linear = [
    (x, yB, 'B — Curved',              '#26A69A'),
    (x, yC, 'C — Threshold',           '#FFA726'),
    (x, yD, 'D — Interaction (x1 only)', '#AB47BC'),
]

for ax, (xi, yi, title, color) in zip(axes, non_linear):
    X_col = xi.reshape(-1, 1)
    lm    = LinearRegression().fit(X_col, yi)
    tree  = DecisionTreeRegressor(max_depth=5, random_state=0).fit(X_col, yi)
    sort_i = xi.argsort()
    xplot  = xi[sort_i]

    ax.scatter(xi, yi, s=10, alpha=0.3, color=color, label='Data')
    ax.plot(xplot, lm.predict(xplot.reshape(-1,1)),
            'k--', lw=2, label=f'Linear  RMSE={rmse(yi, lm.predict(X_col)):.2f}')
    ax.plot(xplot, tree.predict(xplot.reshape(-1,1)),
            color='orangered', lw=2, label=f'Tree    RMSE={rmse(yi, tree.predict(X_col)):.2f}')
    ax.set_xlabel('x')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Linear (dashed) vs Decision Tree (red)\nThe tree follows the data shape; the line cannot', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('B: tree approximates the curve with small steps — RMSE much lower')
print('C: tree nails the jump with one split — RMSE near zero')
print('D: even the tree struggles with x1 alone — interaction needs both features to be useful')

## Key Takeaways

| What you see in the scatter plot | What it means | Model to try |
|---|---|---|
| Points form a tilted straight band | Linear relationship | Linear / Logistic Regression |
| Points form a U-shape or S-curve | Non-linear (curved) | Decision Tree / Random Forest |
| Points cluster in two separate bands | Threshold / step effect | Decision Tree / Random Forest |
| Feature alone shows no clear pattern | Possible interaction with another feature | Tree models; check feature pairs |

### The Residual Test — apply this every time

1. Fit a linear model
2. Plot residuals (actual − predicted) vs each feature
3. If residuals scatter randomly around zero → linear is working
4. If residuals form a shape → switch to a tree model

> Start with linear. If the residuals show structure, the data is telling you to use trees.